In [ ]:
#r "nuget: ScottPlot, 5.0.*"
#!import DefiniteIntegral.cs
using System;
using System.Diagnostics;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;

In [ ]:
double a = -100.0;
double b = 100.0;
Func<double, double> function = Math.Sin;

double[] steps = { 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6 };
double targetAccuracy = 1e-4;
double chosenStep = 1e-1;
bool stepFound = false;

foreach (var step in steps)
{
    double result = DefiniteIntegral.SolveSingleThread(a, b, function, step);
    double error = Math.Abs(result - 0.0);
    if (error <= targetAccuracy && !stepFound)
    {
        chosenStep = step;
        stepFound = true;
    }
}

double workingStep = 1e-5; 
Console.WriteLine($"Выбранный шаг для точности 1e-4: {chosenStep:E0}");

In [ ]:
int maxThreads = Environment.ProcessorCount;
int[] threadCounts = Enumerable.Range(1, maxThreads).ToArray();
double[] averageTimesMs = new double[threadCounts.Length];
int measurementsCount = 5;

DefiniteIntegral.Solve(a, b, function, workingStep, 1);

for (int i = 0; i < threadCounts.Length; i++)
{
    int currentThreads = threadCounts[i];
    double totalTime = 0;

    for (int m = 0; m < measurementsCount; m++)
    {
        Stopwatch sw = Stopwatch.StartNew();
        DefiniteIntegral.Solve(a, b, function, workingStep, currentThreads);
        sw.Stop();
        totalTime += sw.Elapsed.TotalMilliseconds;
    }
    averageTimesMs[i] = totalTime / measurementsCount;
}

int bestIndex = 0;
for (int i = 1; i < averageTimesMs.Length; i++)
{
    if (averageTimesMs[i] < averageTimesMs[bestIndex]) bestIndex = i;
}
int optimalThreads = threadCounts[bestIndex];
double bestParallelTime = averageTimesMs[bestIndex];

var plot = new ScottPlot.Plot();
plot.Title("Зависимость времени вычисления от количества потоков");

double[] xValues = averageTimesMs;
double[] yValues = threadCounts.Select(t => (double)t).ToArray();

var scatter = plot.Add.Scatter(xValues, yValues);
scatter.LineWidth = 2;
scatter.MarkerSize = 8;

plot.XLabel("Время выполнения Solve (мс)");
plot.YLabel("Количество потоков");

plot.SavePng("plot.png", 800, 600);


In [ ]:
double totalSequentialTime = 0;

for (int m = 0; m < measurementsCount; m++)
{
    Stopwatch sw = Stopwatch.StartNew();
    DefiniteIntegral.SolveSingleThread(a, b, function, workingStep);
    sw.Stop();
    totalSequentialTime += sw.Elapsed.TotalMilliseconds;
}

double avgSequentialTime = totalSequentialTime / measurementsCount;
double speedUpPercent = ((avgSequentialTime - bestParallelTime) / avgSequentialTime) * 100;

string report = $"""
Оптимальный размер шага: {workingStep:E3}
Оптимальное количество потоков: {optimalThreads}
Скорость однопоточной версии: {avgSequentialTime:F2} мс
Скорость многопоточной версии: {bestParallelTime:F2} мс
Ускорение: {speedUpPercent:F2}%
""";

Console.WriteLine(report);

File.WriteAllText("results.txt", report);